# Uji Kelayakan Metodologi Bab 3

Notebook ini menguji **setiap komponen Bab 3 yang bisa gagal**, supaya kamu tahu mana yang layak dikerjakan sebelum menulis naskahnya. Bukan pipeline penelitian, melainkan pemeriksaan kelayakan.

Urutannya dari yang paling murah dan paling mendasar ke yang paling mahal. Inferensi lebih dulu, sebab kalau mesinmu tidak sanggup menjalankan model sama sekali, menguji pelatihan LoRA tidak ada gunanya. Pelatihan juga hanya menyangkut satu varian pada rumusan masalah 3, sedangkan inferensi menyangkut keenamnya pada rumusan masalah 1 dan 2.

| Bagian | Menguji | Subbab | Butuh |
|---|---|---|---|
| 1–2 | Lingkungan dan konfigurasi | 3.9 | — |
| **3** | **Inferensi dasar, RTF, dan memori** | 3.7.2, 3.8.3 | satu berkas audio |
| 4 | Penyuntikan daftar istilah | 3.7.3 | audio + daftar istilah |
| 5 | Segmentasi data latih | 3.4.4, 3.7.4 | satu berkas audio |
| 6 | Normalisasi dan pemenggalan teks | 3.6 | — |
| 7 | Penandaan glosarium, WER, CER, TTER | 3.5.4, 3.8.1, 3.8.2 | — |
| 8 | Pemotongan blok untuk uji statistik | 3.10.2 | — |
| 9 | Taksiran memori LoRA | 3.7.4 | — |
| 10–11 | Kelayakan pelatihan LoRA | 3.7.4, 3.8.4 | unduhan model |
| 12 | Penanganan bila memori habis | 3.7.4 | unduhan model |
| 13 | Anggaran waktu keseluruhan | perencanaan | hasil Bagian 3 dan 11 |

**Kalau cuma punya waktu sebentar:** jalankan Bagian 1, 2, lalu 3. Itu sudah menjawab pertanyaan "laptopku sanggup atau tidak" untuk seluruh rumusan masalah 1 dan 2.

Bagian 6, 7, dan 8 berjalan tanpa audio maupun GPU, jadi bisa dikerjakan kapan saja.


## Bagian 1 — Pemasangan dan pemeriksaan lingkungan

In [1]:
# Jalankan sekali. Hapus komentar untuk paket yang belum ada.
# !pip install -q torch transformers peft accelerate psutil jiwer faster-whisper
# Opsional, pengoptimal 8-bit (hanya CUDA):
# !pip install -q bitsandbytes

In [1]:
import os, sys, time, math, gc, re, json, platform, unicodedata
from collections import Counter

import psutil
try:
    import torch
    ADA_TORCH = True
except ImportError:
    ADA_TORCH = False
    print("[!] torch belum terpasang. Bagian 9-7, 8, 11 tidak dapat dijalankan.")

def info_lingkungan():
    d = {
        "sistem_operasi": f"{platform.system()} {platform.release()}",
        "python": sys.version.split()[0],
        "cpu": platform.processor() or platform.machine(),
        "inti_fisik": psutil.cpu_count(logical=False),
        "inti_logis": psutil.cpu_count(logical=True),
        "ram_total_gb": round(psutil.virtual_memory().total / 1024**3, 1),
        "ram_tersedia_gb": round(psutil.virtual_memory().available / 1024**3, 1),
        "gpu": None,
    }
    if ADA_TORCH:
        d["torch"] = torch.__version__
        if torch.cuda.is_available():
            p = torch.cuda.get_device_properties(0)
            d.update(gpu=p.name, vram_gb=round(p.total_memory / 1024**3, 1),
                     cuda=torch.version.cuda, kapabilitas=f"{p.major}.{p.minor}",
                     dukung_bf16=torch.cuda.is_bf16_supported())
    return d

ENV = info_lingkungan()
for k, v in ENV.items():
    print(f"{k:20s}: {v}")

if ENV["gpu"] is None:
    print("\n[!] Tidak ada GPU CUDA. Pelatihan LoRA akan sangat lambat dan RTF akan buruk.")
    print("    Notebook tetap jalan; hasilnya jadi bukti terukur untuk subbab 3.9.1.")

def bersihkan():
    gc.collect()
    if ADA_TORCH and torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

def puncak_memori_gb():
    if ADA_TORCH and torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / 1024**3
    return psutil.Process().memory_info().rss / 1024**3


sistem_operasi      : Windows 11
python              : 3.12.4
cpu                 : AMD64 Family 25 Model 117 Stepping 2, AuthenticAMD
inti_fisik          : 8
inti_logis          : 16
ram_total_gb        : 13.8
ram_tersedia_gb     : 1.8
gpu                 : NVIDIA GeForce RTX 3050 6GB Laptop GPU
torch               : 2.4.1+cu121
vram_gb             : 6.0
cuda                : 12.1
kapabilitas         : 8.6
dukung_bf16         : True


## Bagian 2 — Konfigurasi

Satu tempat untuk semua angka. Nilai bawaan mengikuti Liu, Yang dan Qu (2024) untuk LoRA, dan pembagian data 65 : 10 : 25 dari 12 jam rekaman.

In [2]:
# --- Varian, urut dari kecil ke besar ---------------------------------------
VARIAN_DIUJI = [
    ("tiny",           "openai/whisper-tiny"),
    ("base",           "openai/whisper-base"),
    ("small",          "openai/whisper-small"),
    ("medium",         "openai/whisper-medium"),
    ("large-v3-turbo", "openai/whisper-large-v3-turbo"),
    ("large-v3",       "openai/whisper-large-v3"),
]
HENTI_SETELAH_GAGAL = True

# --- LoRA (Liu, Yang dan Qu, 2024) ------------------------------------------
LORA_R, LORA_ALPHA, LORA_DROPOUT = 32, 64, 0.05
TARGET_MODULES = ["q_proj", "v_proj"]

# --- Pelatihan ---------------------------------------------------------------
BATCH_SIZE, GRAD_ACCUM = 4, 1
PANJANG_LABEL = 128        # perkiraan token per segmen; setel ulang setelah Bagian 7
GRADIENT_CHECKPOINTING = False
PENGOPTIMAL_8BIT = False
EPOCH = 35

# --- Ukuran data --------------------------------------------------------------
JAM_TOTAL, JAM_LATIH, JAM_UJI = 12.05, 7.7, 2.9
RERATA_DETIK_SEGMEN = 20.0

# --- Decoding (Tabel 3.4) -----------------------------------------------------
DECODE = dict(language="id", task="transcribe", beam_size=5,
              vad_filter=True, condition_on_previous_text=False)

# --- Pengukuran ---------------------------------------------------------------
LANGKAH_PEMANASAN, LANGKAH_UKUR = 3, 8
ULANG_RTF = 3
BLOK_ANALISIS_DETIK = 180      # panjang sampel Kuhn, et al. (2024)

# --- Berkas milikmu -----------------------------------------------------------
BERKAS_AUDIO = "contoh.wav"    # satu rekaman, 16 kHz mono, lebih dari 2 menit

if ADA_TORCH:
    PERANGKAT = "cuda" if torch.cuda.is_available() else "cpu"
    DTYPE = (torch.bfloat16 if (PERANGKAT == "cuda" and torch.cuda.is_bf16_supported())
             else torch.float16 if PERANGKAT == "cuda" else torch.float32)
else:
    PERANGKAT, DTYPE = "cpu", None

n_segmen      = int(JAM_LATIH * 3600 / RERATA_DETIK_SEGMEN)
langkah_epoch = math.ceil(n_segmen / (BATCH_SIZE * GRAD_ACCUM))
total_langkah = langkah_epoch * EPOCH

print(f"Perangkat            : {PERANGKAT}   dtype {DTYPE}")
print(f"Segmen latih         : {n_segmen:,}")
print(f"Langkah per epoch    : {langkah_epoch:,}")
print(f"Total langkah        : {total_langkah:,}  ({EPOCH} epoch)")

Perangkat            : cuda   dtype torch.bfloat16
Segmen latih         : 1,386
Langkah per epoch    : 347
Total langkah        : 12,145  (35 epoch)


In [1]:
from transformers.models.whisper.english_normalizer import BasicTextNormalizer
print(BasicTextNormalizer()("Kata-kata itu di-encode oleh encoder-decoder-nya."))

c:\Users\Aufii\Documents\Informatika\OTHERS\8. RISET-SKRIPSI\assistive-communicative\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


kata kata itu di encode oleh encoder decoder nya 


## Bagian 3 — Inferensi dasar, RTF, dan memori (Tahap 1)

**Mulai dari sini.** Ini menjawab pertanyaan paling mendasar: varian mana yang sanggup dijalankan mesinmu, seberapa cepat, dan berapa memori yang dipakainya. Cukup satu berkas audio, tanpa transkrip, tanpa pelatihan apa pun.

Prosedurnya mengikuti subbab 3.8.3: satu jalan pemanasan dibuang, lalu diulang beberapa kali, lalu dilaporkan rerata dan simpangan baku. **Penyebutnya durasi rekaman utuh**, bukan durasi setelah VAD, karena kuliah terus berjalan meskipun pengajar sedang diam. Waktu pemuatan model dikecualikan dari waktu proses, tetapi tetap dicatat sendiri.

Varian diuji dari yang terkecil, jadi kamu dapat jawaban pertama dalam hitungan menit. Kalau satu varian gagal, yang lebih besar dilewati.

Angka dari sini masih boleh kasar. Yang penting: pipeline jalan, dan kamu punya bayangan besaran sebelum menulis subbab 3.9.1.

In [ ]:
import statistics as stat

def ukur_rtf(nama_varian, berkas, ulang=None, hotwords=None, initial_prompt=None):
    from faster_whisper import WhisperModel
    ulang = ULANG_RTF if ulang is None else ulang
    tipe = "float16" if PERANGKAT == "cuda" else "int8"
    hasil = {"varian": nama_varian}
    m = None
    bersihkan()
    dasar_ram = psutil.Process().memory_info().rss / 1024**3
    try:
        t0 = time.time()
        m = WhisperModel(nama_varian, device=PERANGKAT, compute_type=tipe)
        hasil["waktu_muat_detik"] = round(time.time() - t0, 1)

        opsi = dict(DECODE)
        if hotwords:        opsi["hotwords"] = hotwords
        if initial_prompt:  opsi["initial_prompt"] = initial_prompt

        waktu, teks_akhir, info_akhir = [], None, None
        for k in range(ulang + 1):
            t1 = time.time()
            segmen, info = m.transcribe(berkas, **opsi)
            segmen = list(segmen)
            dt = time.time() - t1
            if k > 0: waktu.append(dt)
            teks_akhir, info_akhir = segmen, info

        hasil.update(
            durasi_detik=round(info_akhir.duration, 1),
            durasi_setelah_vad=round(info_akhir.duration_after_vad, 1),
            waktu_proses_rerata=round(stat.mean(waktu), 2),
            waktu_proses_sd=round(stat.stdev(waktu), 2) if len(waktu) > 1 else 0.0,
            rtf_rerata=round(stat.mean(waktu) / info_akhir.duration, 3),
            rtf_sd=round(stat.stdev(waktu) / info_akhir.duration, 4) if len(waktu) > 1 else 0.0,
            jumlah_segmen=len(teks_akhir),
            jumlah_kata=sum(len(s.text.split()) for s in teks_akhir),
            status="BERHASIL",
        )
        if ADA_TORCH and torch.cuda.is_available():
            hasil["puncak_vram_gb"] = round(torch.cuda.max_memory_allocated() / 1024**3, 2)
        else:
            hasil["puncak_ram_gb"] = round(
                psutil.Process().memory_info().rss / 1024**3 - dasar_ram, 2)
        hasil["_segmen"] = teks_akhir
    except Exception as e:
        hasil["status"] = "GAGAL"
        hasil["pesan"] = f"{type(e).__name__}: {str(e).splitlines()[0][:150]}"
    finally:
        del m; bersihkan()
    return hasil


HASIL_RTF = []
if not os.path.exists(BERKAS_AUDIO):
    print(f"[!] {BERKAS_AUDIO} tidak ada. Isi BERKAS_AUDIO di Bagian 2 dengan salah satu rekamanmu.")
else:
    kunci_mem = "puncak_vram_gb" if (ADA_TORCH and torch.cuda.is_available()) else "puncak_ram_gb"
    print(f"{'Varian':<16}{'Muat':>7}{'Proses':>9}{'RTF':>8}{'SD':>8}{'Memori':>9}   Waktu nyata")
    print("-" * 76)
    for nama, _ in VARIAN_DIUJI:
        h = ukur_rtf(nama, BERKAS_AUDIO); HASIL_RTF.append(h)
        if h["status"] == "BERHASIL":
            ok = "memenuhi" if h["rtf_rerata"] <= 1 else "TIDAK"
            print(f"{nama:<16}{h['waktu_muat_detik']:>6.0f}s{h['waktu_proses_rerata']:>8.1f}s"
                  f"{h['rtf_rerata']:>8.3f}{h['rtf_sd']:>8.4f}{h.get(kunci_mem, 0):>8.1f}G   {ok}")
        else:
            print(f"{nama:<16}{h['status']:>32}   {h.get('pesan','')[:40]}")
            if HENTI_SETELAH_GAGAL:
                print("\n  Varian lebih besar dilewati karena pasti ikut gagal."); break

    print(f"\nDurasi berkas uji: {HASIL_RTF[0].get('durasi_detik', 0):.0f} detik "
          f"(setelah VAD {HASIL_RTF[0].get('durasi_setelah_vad', 0):.0f} detik)")
    berhasil = [h for h in HASIL_RTF if h["status"] == "BERHASIL"]
    if berhasil:
        jam_tahap1 = sum(h["rtf_rerata"] for h in berhasil) * JAM_UJI
        print(f"Perkiraan Tahap 1 atas {JAM_UJI} jam data uji, "
              f"{len(berhasil)} varian sekali jalan: {jam_tahap1:.1f} jam")

    lolos = [h for h in HASIL_RTF if h.get("rtf_rerata", 99) <= 1]
    if lolos:
        print(f"\nVarian yang lolos syarat RTF <= 1: {[h['varian'] for h in lolos]}")
        print("Menurut aturan keputusan di subbab 3.7.5, varian terpilih untuk Tahap 2 dan 3")
        print("adalah yang TTER-nya terendah di antara varian ini.")
    else:
        print("\n[!] Tidak ada varian yang lolos RTF <= 1. Siapkan aturan cadangan di subbab 3.7.5.")

Varian             Muat   Proses     RTF      SD   Memori   Waktu nyata
----------------------------------------------------------------------------


c:\Users\Aufii\Documents\Informatika\OTHERS\8. RISET-SKRIPSI\assistive-communicative\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


tiny                 3s    11.2s   0.019  0.0014     0.0G   memenuhi
base                 0s    12.6s   0.022  0.0002     0.0G   memenuhi
small                1s    18.3s   0.032  0.0001     0.0G   memenuhi
medium               2s    44.3s   0.076  0.0009     0.0G   memenuhi
large-v3-turbo       3s    77.8s   0.134  0.0001     0.0G   memenuhi
large-v3             6s   109.0s   0.187  0.0657     0.0G   memenuhi

Durasi berkas uji: 581 detik (setelah VAD 546 detik)
Perkiraan Tahap 1 atas 2.9 jam data uji, 6 varian sekali jalan: 1.4 jam

Varian yang lolos syarat RTF <= 1: ['tiny', 'base', 'small', 'medium', 'large-v3-turbo', 'large-v3']
Menurut aturan keputusan di subbab 3.7.5, varian terpilih untuk Tahap 2 dan 3
adalah yang TTER-nya terendah di antara varian ini.


## Bagian 4 — Uji pendahuluan penyuntikan daftar istilah (Tahap 2)

**Bagian terpenting di notebook ini.** Subbab 2.7.2 menyatakan `initial_prompt` hanya masuk jendela pertama, sehingga pengaruh daftar istilah memudar pada rekaman panjang. Kalau itu dibiarkan, Tahap 2 akan tampak tidak berefek karena sebab teknis, bukan sebab ilmiah, dan kesimpulan rumusan masalah 3 jadi keliru.

Uji ini membandingkan tiga perlakuan pada rekaman yang sama, lalu menghitung kemunculan istilah glosarium **dipilah menurut waktu**: sebelum dan sesudah detik ke-30, yaitu batas jendela pertama.

Yang diharapkan kalau mekanisme `hotwords` bekerja:

- `initial_prompt` menaikkan kemunculan hanya pada 30 detik pertama.
- `hotwords` menaikkan kemunculan **di sepanjang rekaman**.

Kalau `hotwords` ternyata tidak berbeda dari baseline di luar 30 detik pertama, beralihlah ke mekanisme C, yaitu segmentasi manual, dan jalankan Tahap 1 dengan segmentasi yang sama.

In [5]:
h = ukur_rtf("small", BERKAS_AUDIO, ulang=1)
for s in h["_segmen"]:
    print(f"[{s.start:6.1f}] {s.text.strip()}")

[   1.1] Hal semuanya saya Vida Mastrika dalam video sharing tentang analisis dan desain sistem
[   7.0] kali ini saya akan membahas tentang Unified Modeling Language atau UML
[  12.5] dalam video-video selanjutnya saya akan membahas lebih detail tentang UML
[  16.8] yaitu Use Case Diagram, ada Activity Diagram, Sequence Diagram, dan Class Diagram
[  23.0] pada video kali ini saya akan membahas secara umum saja tentang UML
[  27.2] oke, ayo kita langsung ke materi kali ini
[  30.0] Oke, sebelum kita ke Unified Modeling Language, ada baiknya kita tahu apa yang dimaksud dengan Object-Oriented Analysis and Design.
[  55.4] Pada video-video saya sebelumnya, saya sudah menjelaskan Analyst and Design System itu berdasarkan pemerprograman terstruktur ya.
[  63.4] Jadi kita meranjang sistemnya menggunakan Dataflow Diagram.
[  67.8] Kali ini dengan WML, kita akan merancang sistem yang berdasarkan objek orientasi
[  74.2] Setelah itu kita harus menganalisis dan mendesain sistemnya itu berdasarka

In [6]:
GLOSARIUM_UJI = [
    "uml", "behavior diagram", "class diagram",
    "structure diagram", "object oriented analysis and design",
    "object oriented programming", "lifeline",
]
DAFTAR_ISTILAH = ", ".join(GLOSARIUM_UJI)
VARIAN_UJI_PROMPT = "small"

def _rapikan(t):
    t = t.lower().replace("-", " ")
    t = re.sub(r"[^\w\s]", " ", t)
    return " " + re.sub(r"\s+", " ", t).strip() + " "

def hitung_per_jendela(segmen, istilah, batas=30.0):
    urut = sorted(istilah, key=lambda x: len(x.split()), reverse=True)
    awal, sisa = Counter(), Counter()
    for s in segmen:
        teks = _rapikan(s.text)
        ember = awal if s.start < batas else sisa
        for it in urut:
            pola = " " + _rapikan(it).strip() + " "
            n = teks.count(pola)
            if n:
                ember[it] += n
                teks = teks.replace(pola, " ")   # cegah hitung ganda
    return awal, sisa

if not os.path.exists(BERKAS_AUDIO):
    print(f"[!] {BERKAS_AUDIO} tidak ada.")
else:
    perlakuan = {
        "baseline":       dict(),
        "initial_prompt": dict(initial_prompt=DAFTAR_ISTILAH),
        "hotwords":       dict(hotwords=DAFTAR_ISTILAH),
    }
    ringkas = {}
    for label, kw in perlakuan.items():
        h = ukur_rtf(VARIAN_UJI_PROMPT, BERKAS_AUDIO, ulang=1, **kw)
        if h["status"] != "BERHASIL":
            print(f"{label}: {h['status']} - {h.get('pesan','')}"); continue
        awal, sisa = hitung_per_jendela(h["_segmen"], GLOSARIUM_UJI)
        ringkas[label] = dict(awal=sum(awal.values()), sisa=sum(sisa.values()),
                              rtf=h["rtf_rerata"], kata=h["jumlah_kata"], rinci_sisa=sisa)

    print(f"{'Perlakuan':<18}{'<30 dtk':>9}{'>=30 dtk':>10}{'RTF':>9}{'Kata':>8}")
    print("-" * 54)
    for label, r in ringkas.items():
        print(f"{label:<18}{r['awal']:>9}{r['sisa']:>10}{r['rtf']:>9.3f}{r['kata']:>8}")

    if {"baseline", "hotwords"} <= set(ringkas):
        b, hw = ringkas["baseline"], ringkas["hotwords"]
        naik_sisa = hw["sisa"] - b["sisa"]
        print(f"\nSelisih kemunculan di luar 30 detik pertama, hotwords lawan baseline: {naik_sisa:+d}")
        if naik_sisa > 0:
            print("-> Mekanisme A bekerja. Daftar istilah sampai ke jendela-jendela berikutnya.")
            print("   Pakai mekanisme A dan catat hasil uji ini di subbab 3.7.3.")
        else:
            print("-> Belum terbukti. Periksa versi faster-whisper, lalu ulangi dengan istilah")
            print("   yang benar-benar sering salah ditranskripsi. Bila tetap nihil, pakai")
            print("   mekanisme C (segmentasi manual) dan samakan alurnya dengan Tahap 1.")

    if {"baseline", "initial_prompt"} <= set(ringkas):
        ip = ringkas["initial_prompt"]
        print(f"\ninitial_prompt lawan baseline: {ip['awal']-ringkas['baseline']['awal']:+d} "
              f"pada 30 detik pertama, {ip['sisa']-ringkas['baseline']['sisa']:+d} sesudahnya.")
        print("Pola naik di awal tetapi datar sesudahnya persis menegaskan klaim subbab 2.7.2.")

    if "baseline" in ringkas and "hotwords" in ringkas:
        d = ringkas["hotwords"]["rtf"] - ringkas["baseline"]["rtf"]
        print(f"\nSelisih RTF akibat daftar istilah: {d:+.3f}")
        print("Inilah bukti langsung untuk baris 'pengaruh terhadap kecepatan pemrosesan'")
        print("pada Tabel 2.5, yang selama ini baru berupa dugaan dari literatur.")

Perlakuan           <30 dtk  >=30 dtk      RTF    Kata
------------------------------------------------------
baseline                  4        22    0.035    1045
initial_prompt            4        20    0.091    1049
hotwords                  4        27    0.094    1058

Selisih kemunculan di luar 30 detik pertama, hotwords lawan baseline: +5
-> Mekanisme A bekerja. Daftar istilah sampai ke jendela-jendela berikutnya.
   Pakai mekanisme A dan catat hasil uji ini di subbab 3.7.3.

initial_prompt lawan baseline: +0 pada 30 detik pertama, -2 sesudahnya.
Pola naik di awal tetapi datar sesudahnya persis menegaskan klaim subbab 2.7.2.

Selisih RTF akibat daftar istilah: +0.059
Inilah bukti langsung untuk baris 'pengaruh terhadap kecepatan pemrosesan'
pada Tabel 2.5, yang selama ini baru berupa dugaan dari literatur.


## Bagian 5 — Segmentasi data latih (subbab 3.4.4 dan 3.7.4)

Whisper hanya memproses audio maksimum 30 detik (Radford, et al., 2023), jadi rekaman puluhan menit harus dipotong dan dipasangkan dengan transkripnya.

Ada jalan pintas yang menghemat banyak sekali waktumu. Karena transkrip acuan disusun dengan mengoreksi keluaran Whisper, dan keluaran itu sudah bersegmen lengkap dengan stempel waktu, **transkrip acuanmu otomatis sudah tersegmentasi** kalau kamu mengoreksinya per segmen sejak awal. Tidak perlu penyelarasan paksa terpisah.

Sel ini memeriksa apakah segmen yang dihasilkan memang di bawah 30 detik, dan berapa banyak segmen yang akan kamu dapat dari 7,7 jam data latih.

In [7]:
if not os.path.exists(BERKAS_AUDIO):
    print(f"[!] {BERKAS_AUDIO} tidak ada.")
else:
    h = ukur_rtf("small", BERKAS_AUDIO, ulang=1)
    if h["status"] != "BERHASIL":
        print("Gagal:", h.get("pesan"))
    else:
        seg = h["_segmen"]
        durasi = [s.end - s.start for s in seg]
        kata   = [len(s.text.split()) for s in seg]
        durasi_urut = sorted(durasi)
        tengah = durasi_urut[len(durasi_urut)//2]
        print(f"Jumlah segmen        : {len(seg)}")
        print(f"Durasi segmen        : min {min(durasi):.1f}s, median {tengah:.1f}s, "
              f"maks {max(durasi):.1f}s")
        print(f"Melebihi 30 detik    : {sum(1 for d in durasi if d > 30.0)}")
        print(f"Kata per segmen      : min {min(kata)}, maks {max(kata)}, "
              f"rerata {sum(kata)/len(kata):.1f}")

        rerata = sum(durasi) / len(durasi)
        n_est = int(JAM_LATIH * 3600 / rerata)
        print(f"\nRerata durasi segmen : {rerata:.1f} detik")
        print(f"Perkiraan segmen dari {JAM_LATIH} jam data latih: {n_est:,}")
        print(f"Nilai RERATA_DETIK_SEGMEN di Bagian 2 saat ini {RERATA_DETIK_SEGMEN}; "
              f"ganti menjadi {rerata:.1f} lalu jalankan ulang Bagian 2 dan 5.")

        # Perkiraan panjang label untuk Bagian 10
        try:
            from transformers import WhisperTokenizer
            tok = WhisperTokenizer.from_pretrained("openai/whisper-small",
                                                   language="indonesian", task="transcribe")
            panjang = [len(tok.encode(s.text)) for s in seg]
            pj = sorted(panjang)
            print(f"\nToken per segmen     : median {pj[len(pj)//2]}, "
                  f"persentil 95 {pj[int(len(pj)*0.95)]}, maks {max(panjang)}")
            print(f"PANJANG_LABEL di Bagian 2 saat ini {PANJANG_LABEL}; "
                  f"nilai yang wajar sekitar {pj[int(len(pj)*0.95)]}.")
        except Exception as e:
            print("Tokenizer tidak dapat dimuat:", e)

Jumlah segmen        : 86
Durasi segmen        : min 1.6s, median 6.0s, maks 25.3s
Melebihi 30 detik    : 0
Kata per segmen      : min 4, maks 30, rerata 12.2

Rerata durasi segmen : 6.6 detik
Perkiraan segmen dari 7.7 jam data latih: 4,212
Nilai RERATA_DETIK_SEGMEN di Bagian 2 saat ini 20.0; ganti menjadi 6.6 lalu jalankan ulang Bagian 2 dan 5.

Token per segmen     : median 25, persentil 95 47, maks 70
PANJANG_LABEL di Bagian 2 saat ini 128; nilai yang wajar sekitar 47.


## Bagian 6 — Normalisasi dan pemenggalan teks (subbab 3.6)

Tidak butuh audio maupun GPU. Kerjakan ini lebih dulu.

Yang diuji: apakah aturanmu berperilaku seperti yang kamu kira pada teks bahasa Indonesia bercampur istilah Inggris. Dua hal yang paling sering meleset adalah **tanda hubung** dan **bentuk ulang**, sebab keduanya mengubah jumlah kata, dan jumlah kata adalah penyebut WER.

In [ ]:
def normalisasi(teks, tingkat="dasar", pisah_hubung=False):
    """tingkat: 'mentah', 'dasar', atau 'lanjut'."""
    t = unicodedata.normalize("NFKC", teks)
    if tingkat == "mentah":
        return t.strip()

    t = t.lower()
    t = re.sub(r"[\u2010-\u2015]", "-", t)
    t = re.sub(r"[\u2018\u2019\u201c\u201d]", "", t)

    if tingkat == "lanjut":
        singkatan = {r"\byg\b": "yang", r"\bdgn\b": "dengan",
                     r"\btdk\b": "tidak", r"\bkrn\b": "karena"}
        for pola, ganti in singkatan.items():
            t = re.sub(pola, ganti, t)
        lisan = {r"\bnggak\b": "tidak", r"\bgak\b": "tidak", r"\budah\b": "sudah"}
        for pola, ganti in lisan.items():
            t = re.sub(pola, ganti, t)

    t = re.sub(r"[^\w\s-]", " ", t)
    if pisah_hubung:
        t = t.replace("-", " ")
    else:
        t = re.sub(r"(?<!\w)-|-(?!\w)", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def tokenkan(teks):
    return teks.split()

UJI = [
    "Jadi overfitting itu terjadi kalau modelnya terlalu menghafal training set.",
    "Kata-kata di dalam korpus itu perlu di-preprocessing dulu, ya.",
    "Nanti hasilnya di-encode, lalu masuk ke encoder-decoder-nya.",
    "Ada 3 lapisan: input layer, hidden layer, sama output layer.",
    "Yg penting model-nya nggak overfitting; udah gitu aja.",
]

print("Pengaruh perlakuan tanda hubung terhadap jumlah kata\n")
print(f"{'Kalimat':<10}{'mentah':>8}{'dasar (-)':>11}{'dasar (dipisah)':>17}{'lanjut':>9}")
print("-" * 56)
for i, s in enumerate(UJI, 1):
    a = len(tokenkan(normalisasi(s, "mentah")))
    b = len(tokenkan(normalisasi(s, "dasar", pisah_hubung=False)))
    cc = len(tokenkan(normalisasi(s, "dasar", pisah_hubung=True)))
    d = len(tokenkan(normalisasi(s, "lanjut", pisah_hubung=False)))
    print(f"{i:<10}{a:>8}{b:>11}{cc:>17}{d:>9}")

print("\nContoh keluaran per tingkat:\n")
for s in UJI[1:3]:
    print(f"  asli   : {s}")
    print(f"  dasar  : {normalisasi(s, 'dasar')}")
    print(f"  lanjut : {normalisasi(s, 'lanjut')}\n")

print("Yang harus kamu putuskan dan tuliskan di subbab 3.6:")
print("  1. Tanda hubung dipertahankan atau dipisah. Selisihnya terlihat di kolom 2 dan 3.")
print("  2. Bentuk berimbuhan seperti 'di-encode' dihitung satu kata atau dua.")
print("     Keputusan ini juga menentukan penandaan glosarium di Bagian 7.")
print("  3. Isi kamus singkatan dan bentuk lisan untuk tingkat lanjut.")

Pengaruh perlakuan tanda hubung terhadap jumlah kata

Kalimat     mentah  dasar (-)  dasar (dipisah)   lanjut
--------------------------------------------------------
1               10         10               10       10
2                9          9               11        9
3                7          7               10        7
4               10         10               10       10
5                8          8                9        8

Contoh keluaran per tingkat:

  asli   : Kata-kata di dalam korpus itu perlu di-preprocessing dulu, ya.
  dasar  : kata-kata di dalam korpus itu perlu di-preprocessing dulu ya
  lanjut : kata-kata di dalam korpus itu perlu di-preprocessing dulu ya

  asli   : Nanti hasilnya di-encode, lalu masuk ke encoder-decoder-nya.
  dasar  : nanti hasilnya di-encode lalu masuk ke encoder-decoder-nya
  lanjut : nanti hasilnya di-encode lalu masuk ke encoder-decoder-nya

Yang harus kamu putuskan dan tuliskan di subbab 3.6:
  1. Tanda hubung dipertahankan atau 

## Bagian 7 — Penandaan glosarium, WER, CER, dan TTER

Inti perhitungan Bab 4, dan bisa kamu uji sekarang juga tanpa audio.

Isinya: penyelarasan jarak sunting yang menghasilkan himpunan operasi **E** (Persamaan 2.2), lalu tiga cara menyaring himpunan yang sama itu — WER (Persamaan 2.3), CER (Persamaan 2.4), dan TTER (Persamaan 2.7 dan 2.8).

Dua hal yang diperiksa di sini:

1. **Urutan operasi.** Normalisasi dijalankan lebih dulu, penandaan glosarium menyusul di atas token hasil normalisasi. Kalau dibalik, indeks pada himpunan G bergeser dan TTER-nya salah tanpa gejala apa pun. Ini titik paling rawan di seluruh perhitungan.
2. **Kesahihan implementasi.** WER dari penyelarasanmu sendiri dibandingkan dengan `jiwer`. Kalau keduanya sama, penyelarasannya benar, dan TTER yang menumpang penyelarasan itu ikut bisa dipercaya.

In [ ]:
def selaraskan(acuan, hipotesis):
    n, m = len(acuan), len(hipotesis)
    D = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1): D[i][0] = i
    for j in range(m + 1): D[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            D[i][j] = (D[i-1][j-1] if acuan[i-1] == hipotesis[j-1]
                       else 1 + min(D[i-1][j-1], D[i-1][j], D[i][j-1]))
    ops, i, j = [], n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and acuan[i-1] == hipotesis[j-1] and D[i][j] == D[i-1][j-1]:
            i, j = i - 1, j - 1
        elif i > 0 and j > 0 and D[i][j] == D[i-1][j-1] + 1:
            ops.append(("substitusi", i - 1, j - 1)); i, j = i - 1, j - 1
        elif i > 0 and D[i][j] == D[i-1][j] + 1:
            ops.append(("penghapusan", i - 1, None)); i -= 1
        else:
            ops.append(("penyisipan", i, j - 1)); j -= 1
    return ops[::-1]

def hitung_wer(acuan, hipotesis):
    ops = selaraskan(acuan, hipotesis)
    return len(ops) / len(acuan), Counter(o[0] for o in ops)

def hitung_cer(acuan_teks, hipotesis_teks):
    a, h = list(acuan_teks.replace(" ", "")), list(hipotesis_teks.replace(" ", ""))
    return len(selaraskan(a, h)) / len(a)

def tandai_glosarium(token_acuan, glosarium):
    urut = sorted([tokenkan(normalisasi(g, "dasar")) for g in glosarium],
                  key=len, reverse=True)
    G, i, n = set(), 0, len(token_acuan)
    while i < n:
        cocok = next((len(g) for g in urut if token_acuan[i:i+len(g)] == g), 0)
        if cocok:
            G.update(range(i, i + cocok)); i += cocok
        else:
            i += 1
    return G

def hitung_tter(acuan, hipotesis, G):
    if not G:
        return None, 0, 0
    ops, n = selaraskan(acuan, hipotesis), len(acuan)
    tepi = (n - 1) in G
    EG = [o for o in ops if o[1] in G or (tepi and o[1] >= n)]
    return len(EG) / len(G), len(EG), len(G)

def kumpulkan_tter(pasangan, glosarium):
    pemb = penye = 0; dipakai = dilewati = 0
    for acuan_teks, hip_teks in pasangan:
        a = tokenkan(normalisasi(acuan_teks, "dasar"))
        h = tokenkan(normalisasi(hip_teks, "dasar"))
        G = tandai_glosarium(a, glosarium)
        if not G:
            dilewati += 1; continue
        _, eg, g = hitung_tter(a, h, G)
        pemb += eg; penye += g; dipakai += 1
    return (pemb / penye if penye else None), pemb, penye, dipakai, dilewati

print("Fungsi metrik siap.")

Fungsi metrik siap.


In [ ]:
GLOSARIUM = ["overfitting", "training set", "machine learning", "encoder", "decoder"]

PASANGAN = [
    ("jadi overfitting itu terjadi kalau modelnya terlalu menghafal training set",
     "jadi over fitting itu terjadi kalau modelnya terlalu menghafal training sheet"),
    ("machine learning butuh data yang banyak sekali",
     "machine learning butuh data yang banyak sekali"),
    ("hasilnya masuk ke encoder lalu keluar dari decoder",
     "hasilnya masuk ke enkoder lalu keluar dari dekoder"),
    ("kita bahas dulu konsep dasarnya sebelum masuk ke rumus",
     "kita bahas dulu konsep dasarnya sebelum masuk ke rumus"),
]

print(f"{'#':<3}{'WER':>8}{'CER':>8}{'TTER':>9}{'|E_G|':>7}{'|G|':>6}   Catatan")
print("-" * 66)
for i, (ref, hyp) in enumerate(PASANGAN, 1):
    a = tokenkan(normalisasi(ref, "dasar"))
    h = tokenkan(normalisasi(hyp, "dasar"))
    G = tandai_glosarium(a, GLOSARIUM)
    wer, _ = hitung_wer(a, h)
    cer = hitung_cer(normalisasi(ref, "dasar"), normalisasi(hyp, "dasar"))
    t, eg, g = hitung_tter(a, h, G)
    if t is None:
        print(f"{i:<3}{wer:>8.3f}{cer:>8.3f}{'-':>9}{'-':>7}{'-':>6}   G kosong, dikeluarkan")
    else:
        istilah = [a[k] for k in sorted(G)]
        print(f"{i:<3}{wer:>8.3f}{cer:>8.3f}{t:>9.3f}{eg:>7}{g:>6}   G = {istilah}")

tter, pemb, penye, dipakai, dilewati = kumpulkan_tter(PASANGAN, GLOSARIUM)
wer_gab = (sum(len(selaraskan(tokenkan(normalisasi(r,'dasar')), tokenkan(normalisasi(hh,'dasar'))))
               for r, hh in PASANGAN) /
           sum(len(tokenkan(normalisasi(r,'dasar'))) for r, _ in PASANGAN))
print(f"\nAgregat  WER {wer_gab:.3f}   TTER {tter:.3f}  ({pemb}/{penye})")
print(f"Segmen dipakai {dipakai}, dikeluarkan karena G kosong {dilewati}")
print("\nPerhatikan: WER dan TTER tidak sebanding langsung, sebab penyebutnya berbeda")
print("(subbab 2.6.4). Yang sah dibandingkan adalah antarkonfigurasi pada metrik yang sama.")

#       WER     CER     TTER  |E_G|   |G|   Catatan
------------------------------------------------------------------
1     0.300   0.031    1.000      3     3   G = ['overfitting', 'training', 'set']
2     0.000   0.000    0.000      0     2   G = ['machine', 'learning']
3     0.250   0.047    1.000      2     2   G = ['encoder', 'decoder']
4     0.000   0.000        -      -     -   G kosong, dikeluarkan

Agregat  WER 0.147   TTER 0.714  (5/7)
Segmen dipakai 3, dikeluarkan karena G kosong 1

Perhatikan: WER dan TTER tidak sebanding langsung, sebab penyebutnya berbeda
(subbab 2.6.4). Yang sah dibandingkan adalah antarkonfigurasi pada metrik yang sama.


In [ ]:
try:
    import jiwer
    print(f"{'#':<3}{'WER sendiri':>13}{'WER jiwer':>12}{'CER sendiri':>13}{'CER jiwer':>12}   Cocok")
    print("-" * 66)
    semua_cocok = True
    for i, (ref, hyp) in enumerate(PASANGAN, 1):
        rn, hn = normalisasi(ref, "dasar"), normalisasi(hyp, "dasar")
        w1, _ = hitung_wer(tokenkan(rn), tokenkan(hn))
        w2 = jiwer.wer(rn, hn)
        c1 = hitung_cer(rn, hn)
        c2 = jiwer.cer(rn.replace(" ", ""), hn.replace(" ", ""))
        ok = abs(w1 - w2) < 1e-9 and abs(c1 - c2) < 1e-9
        semua_cocok &= ok
        print(f"{i:<3}{w1:>13.6f}{w2:>12.6f}{c1:>13.6f}{c2:>12.6f}   {'ya' if ok else 'TIDAK'}")
    print("\n" + ("Penyelarasan terverifikasi. TTER menumpang penyelarasan yang sama,"
                  "\njadi implementasinya ikut dapat dipercaya. Catat hasil ini di subbab 3.8.2."
                  if semua_cocok else
                  "[!] Ada selisih. Periksa aturan normalisasi sebelum melanjutkan."))
except ImportError:
    print("jiwer belum terpasang: pip install jiwer")

#    WER sendiri   WER jiwer  CER sendiri   CER jiwer   Cocok
------------------------------------------------------------------
1       0.300000    0.300000     0.030769    0.030769   ya
2       0.000000    0.000000     0.000000    0.000000   ya
3       0.250000    0.250000     0.046512    0.046512   ya
4       0.000000    0.000000     0.000000    0.000000   ya

Penyelarasan terverifikasi. TTER menumpang penyelarasan yang sama,
jadi implementasinya ikut dapat dipercaya. Catat hasil ini di subbab 3.8.2.


In [ ]:
a = tokenkan("model ini memakai encoder")
h = tokenkan("model ini memakai encoder tambahan lagi")
G = tandai_glosarium(a, ["encoder"])
print("acuan     :", a)
print("hipotesis :", h)
print("G         :", sorted(G), "->", [a[k] for k in sorted(G)])
print("indeks terakhir acuan ada di G:", (len(a) - 1) in G)
t, eg, g = hitung_tter(a, h, G)
print(f"\nTTER dengan aturan tepi   : {t:.3f}  ({eg}/{g})")
EG_tanpa = [o for o in selaraskan(a, h) if o[1] in G]
print(f"TTER tanpa aturan tepi    : {len(EG_tanpa)/len(G):.3f}  ({len(EG_tanpa)}/{g})")
print("\nSelisih ini yang dimaksud aturan tepi. Pastikan implementasi finalmu memakainya.")
print("\nPerhatikan nilainya melebihi 1. Itu wajar dan bukan kekeliruan: penyebutnya")
print("jumlah istilah glosarium, sedangkan pembilangnya bisa memuat penyisipan yang")
print("tak terbatas. WER dan PIER punya sifat yang sama. Sebutkan hal ini di subbab 3.8.2")
print("supaya nilai TTER di atas 100 persen pada Bab 4 tidak dikira salah hitung.")

a2 = tokenkan("kita pakai machine learning untuk itu")
G2 = tandai_glosarium(a2, ["machine learning"])
print(f"\n'machine learning' menempati indeks {sorted(G2)}, "
      f"menyumbang {len(G2)} satuan pada penyebut.")
print("Sesuai ketentuan subbab 2.6.4, sehingga satuan hitung TTER sebanding dengan WER.")

acuan     : ['model', 'ini', 'memakai', 'encoder']
hipotesis : ['model', 'ini', 'memakai', 'encoder', 'tambahan', 'lagi']
G         : [3] -> ['encoder']
indeks terakhir acuan ada di G: True

TTER dengan aturan tepi   : 2.000  (2/1)
TTER tanpa aturan tepi    : 0.000  (0/1)

Selisih ini yang dimaksud aturan tepi. Pastikan implementasi finalmu memakainya.

Perhatikan nilainya melebihi 1. Itu wajar dan bukan kekeliruan: penyebutnya
jumlah istilah glosarium, sedangkan pembilangnya bisa memuat penyisipan yang
tak terbatas. WER dan PIER punya sifat yang sama. Sebutkan hal ini di subbab 3.8.2
supaya nilai TTER di atas 100 persen pada Bab 4 tidak dikira salah hitung.

'machine learning' menempati indeks [2, 3], menyumbang 2 satuan pada penyebut.
Sesuai ketentuan subbab 2.6.4, sehingga satuan hitung TTER sebanding dengan WER.


## Bagian 8 — Pemotongan blok untuk uji statistik (subbab 3.10.2)

Data ujimu sekitar 2 jam 55 menit yang tersebar di 9 sampai 10 rekaman. Uji berpasangan dengan n=10 dayanya lemah: selisih yang nyata bisa gagal terdeteksi hanya karena sampelnya sedikit. Kuhn, et al. (2024) punya 120 sampel.

Jalan keluarnya memotong data uji menjadi blok waktu sebagai satuan analisis. Panjang 3 menit dipilih mengikuti sampel Kuhn, et al. (2024), jadi ada preseden yang bisa disitasi.

**Aturan yang mengikat: transkripsi tetap dijalankan atas rekaman utuh.** Pemotongan hanya dilakukan sesudah penyelarasan, dengan membagi himpunan operasi sunting menurut waktu. Kalau audionya yang dipotong lebih dulu, persoalan jendela pertama pada Bagian 4 ikut hilang — padahal itu justru yang mau diuji.

In [ ]:
DURASI_UJI = [19*60+41, 56*60+14, 30*60+56, 13*60+18, 18*60+42, 35*60+54]
JUMLAH_REKAMAN_UJI = 9

for blok in (120, 180, 300):
    n = sum(max(1, round(d / blok)) for d in DURASI_UJI)
    print(f"Blok {blok//60} menit -> {n:3d} satuan analisis")

n180 = sum(max(1, round(d / BLOK_ANALISIS_DETIK)) for d in DURASI_UJI)
print(f"\nDengan blok {BLOK_ANALISIS_DETIK//60} menit: {n180} satuan, "
      f"dibanding {JUMLAH_REKAMAN_UJI} kalau satuannya per rekaman.")
print("Cukup untuk uji berpasangan dengan daya yang wajar.")

def bagi_ops_per_blok(ops, waktu_kata_acuan, blok=None):
    blok = BLOK_ANALISIS_DETIK if blok is None else blok
    n = len(waktu_kata_acuan)
    ember = {}
    for jenis, i_src, i_res in ops:
        idx = min(i_src, n - 1)
        b = int(waktu_kata_acuan[idx] // blok)
        ember.setdefault(b, []).append((jenis, i_src, i_res))
    return ember

# Peragaan dengan data tiruan
import random
random.seed(0)
waktu_tiruan = sorted(random.uniform(0, 600) for _ in range(1200))   # 1200 kata dalam 10 menit
ops_tiruan = [("substitusi", random.randrange(1200), 0) for _ in range(80)]
ember = bagi_ops_per_blok(ops_tiruan, waktu_tiruan)
print(f"\nPeragaan: 80 operasi sunting tersebar ke {len(ember)} blok "
      f"{BLOK_ANALISIS_DETIK//60} menit")
for b in sorted(ember):
    print(f"  blok {b} (menit {b*BLOK_ANALISIS_DETIK//60}-"
          f"{(b+1)*BLOK_ANALISIS_DETIK//60}): {len(ember[b])} operasi")
print("\nPenyebut tiap blok dihitung dari jumlah kata acuan yang jatuh di blok itu,")
print("dan untuk TTER dari jumlah istilah glosarium di blok itu.")

Blok 2 menit ->  87 satuan analisis
Blok 3 menit ->  58 satuan analisis
Blok 5 menit ->  35 satuan analisis

Dengan blok 3 menit: 58 satuan, dibanding 9 kalau satuannya per rekaman.
Cukup untuk uji berpasangan dengan daya yang wajar.

Peragaan: 80 operasi sunting tersebar ke 4 blok 3 menit
  blok 0 (menit 0-3): 34 operasi
  blok 1 (menit 3-6): 17 operasi
  blok 2 (menit 6-9): 21 operasi
  blok 3 (menit 9-12): 8 operasi

Penyebut tiap blok dihitung dari jumlah kata acuan yang jatuh di blok itu,
dan untuk TTER dari jumlah istilah glosarium di blok itu.


## Bagian 9 — Taksiran memori LoRA secara analitik

Saringan awal supaya kamu tidak mengunduh model 6 GB yang sudah pasti gagal. Angka resminya dari Bagian 11.

In [16]:
PARAMETER_JUTA = {"tiny": 39, "base": 74, "small": 244, "medium": 769,
                  "large-v3-turbo": 809, "large-v3": 1550}

def taksir_memori(nama, dtype_bytes, batch):
    p  = PARAMETER_JUTA[nama] * 1e6
    pl = p * 0.0067                                   # nisbah Liu, Yang dan Qu (2024)
    return {"bobot beku": p * dtype_bytes, "LoRA": pl * 4, "gradien": pl * 4,
            "status pengoptimal": pl * 8,
            "aktivasi": p * dtype_bytes * batch * 0.55}

if ADA_TORCH:
    db = 2 if DTYPE in (torch.float16, torch.bfloat16) else 4
    batas = ENV.get("vram_gb") if PERANGKAT == "cuda" else ENV["ram_tersedia_gb"]
    gb = lambda x: x / 1024**3
    print(f"Taksiran, batch {BATCH_SIZE}, presisi {db*8}-bit. Kapasitas {batas} GB\n")
    print(f"{'Varian':<16}{'Bobot':>9}{'Aktivasi':>10}{'TOTAL':>9}   Status")
    print("-" * 62)
    for nama, _ in VARIAN_DIUJI:
        m = taksir_memori(nama, db, BATCH_SIZE); tot = gb(sum(m.values()))
        st = "muat" if tot < batas * 0.85 else ("ketat" if tot < batas else "TIDAK MUAT")
        print(f"{nama:<16}{gb(m['bobot beku']):>8.2f}G{gb(m['aktivasi']):>9.2f}G{tot:>8.2f}G   {st}")

Taksiran, batch 4, presisi 16-bit. Kapasitas 6.0 GB

Varian              Bobot  Aktivasi    TOTAL   Status
--------------------------------------------------------------
tiny                0.07G     0.16G    0.24G   muat
base                0.14G     0.30G    0.45G   muat
small               0.45G     1.00G    1.48G   muat
medium              1.43G     3.15G    4.66G   muat
large-v3-turbo      1.51G     3.32G    4.90G   muat
large-v3            2.89G     6.35G    9.39G   TIDAK MUAT


## Bagian 10 — Fungsi pengukuran LoRA

In [17]:
def uji_lora(nama, model_id, batch=None, grad_ckpt=None, optim8bit=None):
    """Pasang LoRA, jalankan beberapa langkah latih dengan data tiruan, ukur."""
    from transformers import WhisperForConditionalGeneration
    from peft import LoraConfig, get_peft_model

    batch     = BATCH_SIZE if batch is None else batch
    grad_ckpt = GRADIENT_CHECKPOINTING if grad_ckpt is None else grad_ckpt
    optim8bit = PENGOPTIMAL_8BIT if optim8bit is None else optim8bit
    hasil = {"varian": nama, "batch": batch, "grad_ckpt": grad_ckpt, "optim8bit": optim8bit}

    model = opt = None
    bersihkan(); t0 = time.time()
    try:
        model = WhisperForConditionalGeneration.from_pretrained(model_id, torch_dtype=DTYPE)
        model.config.forced_decoder_ids = None
        model.config.use_cache = False
        hasil["n_mels"] = model.config.num_mel_bins       # 80 varian awal, 128 large-v3 dan turbo
        hasil["waktu_muat_detik"] = round(time.time() - t0, 1)

        model = get_peft_model(model, LoraConfig(
            r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
            target_modules=TARGET_MODULES, bias="none"))

        for _, p in model.named_parameters():             # LoRA tetap 32-bit agar stabil
            if p.requires_grad:
                p.data = p.data.float()

        terlatih = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total    = sum(p.numel() for p in model.parameters())
        hasil.update(parameter_terlatih=terlatih, parameter_total=total,
                     persen_terlatih=round(100 * terlatih / total, 3))

        model.to(PERANGKAT)
        if grad_ckpt:
            model.gradient_checkpointing_enable()
            try: model.enable_input_require_grads()
            except AttributeError: pass

        if optim8bit:
            import bitsandbytes as bnb
            opt = bnb.optim.AdamW8bit([p for p in model.parameters() if p.requires_grad], lr=1e-5)
        else:
            opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-5)

        if DTYPE == torch.float16:
            try: skaler = torch.amp.GradScaler("cuda")
            except (AttributeError, TypeError): skaler = torch.cuda.amp.GradScaler()
        else:
            skaler = None

        v, nm = model.config.vocab_size, hasil["n_mels"]

        def satu_langkah():
            fitur = torch.randn(batch, nm, 3000, dtype=DTYPE, device=PERANGKAT)
            label = torch.randint(0, v - 100, (batch, PANJANG_LABEL), device=PERANGKAT)
            with torch.autocast(device_type=PERANGKAT, dtype=DTYPE, enabled=(PERANGKAT == "cuda")):
                rugi = model(input_features=fitur, labels=label).loss
            if skaler is not None:
                skaler.scale(rugi).backward(); skaler.step(opt); skaler.update()
            else:
                rugi.backward(); opt.step()
            opt.zero_grad(set_to_none=True)

        for _ in range(LANGKAH_PEMANASAN): satu_langkah()
        if torch.cuda.is_available():
            torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()

        t1 = time.time()
        for _ in range(LANGKAH_UKUR): satu_langkah()
        if torch.cuda.is_available(): torch.cuda.synchronize()

        hasil["detik_per_langkah"] = round((time.time() - t1) / LANGKAH_UKUR, 3)
        hasil["puncak_memori_gb"]  = round(puncak_memori_gb(), 2)
        hasil["status"] = "BERHASIL"

    except Exception as e:
        habis = "out of memory" in str(e).lower()
        hasil["status"] = "MEMORI HABIS" if habis else "GAGAL"
        hasil["pesan"]  = f"{type(e).__name__}: {str(e).splitlines()[0][:150]}"
    finally:
        del model, opt; bersihkan()
    return hasil

def ekstrapolasi(dps, batch=None):
    batch = BATCH_SIZE if batch is None else batch
    lintasan = math.ceil(n_segmen / batch) * EPOCH
    return {"jam_total": round(dps * lintasan / 3600, 2),
            "jam_per_epoch": round(dps * math.ceil(n_segmen / batch) / 3600, 2)}

print("Fungsi siap.")

Fungsi siap.


## Bagian 11 — Kelayakan pelatihan LoRA (Tahap 3)

Mengunduh dan menguji tiap varian berurutan. Hasilnya disimpan supaya tidak perlu diulang.

In [18]:
HASIL_LORA = []
for nama, model_id in VARIAN_DIUJI:
    print(f"\n{'='*66}\nMenguji {nama}")
    h = uji_lora(nama, model_id); HASIL_LORA.append(h)
    if h["status"] == "BERHASIL":
        h.update(ekstrapolasi(h["detik_per_langkah"]))
        print(f"  parameter terlatih : {h['parameter_terlatih']:,} ({h['persen_terlatih']}%)")
        print(f"  pita Mel           : {h['n_mels']}")
        print(f"  puncak memori      : {h['puncak_memori_gb']} GB")
        print(f"  detik per langkah  : {h['detik_per_langkah']}")
        print(f"  taksiran waktu     : {h['jam_per_epoch']} jam/epoch, {h['jam_total']} jam total")
    else:
        print(f"  {h['status']}: {h.get('pesan','')}")
        if HENTI_SETELAH_GAGAL:
            print("\n  Varian lebih besar dilewati. Coba Bagian 12."); break

json.dump({"lingkungan": ENV, "lora": HASIL_LORA},
          open("hasil_uji_bab3.json", "w"), indent=2, default=str)

print(f"\n{'Varian':<16}{'Status':<14}{'Par.terlatih':>14}{'Memori':>9}{'dtk/lgk':>9}{'Jam':>9}")
print("-" * 72)
for h in HASIL_LORA:
    if h["status"] == "BERHASIL":
        print(f"{h['varian']:<16}{h['status']:<14}{h['parameter_terlatih']:>13,}"
              f"{h['puncak_memori_gb']:>8.1f}G{h['detik_per_langkah']:>9.2f}{h['jam_total']:>9.1f}")
    else:
        print(f"{h['varian']:<16}{h['status']:<14}" + "".join(f"{'-':>9}" for _ in range(4)))

layak = [h for h in HASIL_LORA if h["status"] == "BERHASIL"]
if layak:
    t = layak[-1]
    print(f"\nVarian terbesar yang dapat dilatih di sini: {t['varian']} ({t['jam_total']} jam)")
    if t["jam_total"] > 24:
        print("[!] Lebih dari sehari. Turunkan epoch, atau latih di GPU sewaan")
        print("    dan laporkan sebagai tuntutan sumber daya (subbab 3.8.4).")


Menguji tiny


c:\Users\Aufii\Documents\Informatika\OTHERS\8. RISET-SKRIPSI\assistive-communicative\.venv\Lib\site-packages\transformers\models\whisper\modeling_whisper.py:599: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


  parameter terlatih : 589,824 (1.538%)
  pita Mel           : 80
  puncak memori      : 0.7 GB
  detik per langkah  : 0.353
  taksiran waktu     : 0.03 jam/epoch, 1.19 jam total

Menguji base
  parameter terlatih : 1,179,648 (1.599%)
  pita Mel           : 80
  puncak memori      : 1.13 GB
  detik per langkah  : 0.935
  taksiran waktu     : 0.09 jam/epoch, 3.15 jam total

Menguji small
  parameter terlatih : 3,538,944 (1.443%)
  pita Mel           : 80
  puncak memori      : 2.93 GB
  detik per langkah  : 0.835
  taksiran waktu     : 0.08 jam/epoch, 2.82 jam total

Menguji medium
  parameter terlatih : 9,437,184 (1.22%)
  pita Mel           : 80
  puncak memori      : 7.49 GB
  detik per langkah  : 11.851
  taksiran waktu     : 1.14 jam/epoch, 39.98 jam total

Menguji large-v3-turbo
  parameter terlatih : 6,553,600 (0.804%)
  pita Mel           : 128
  puncak memori      : 9.13 GB
  detik per langkah  : 22.346
  taksiran waktu     : 2.15 jam/epoch, 75.39 jam total

Menguji large-v3
  

## Bagian 12 — Penanganan bila memori habis (opsional)

Gradient checkpointing dan batch kecil dengan akumulasi gradien **tidak mengubah hasil pelatihan**, hanya memindahkan beban dari memori ke waktu. Keduanya aman dipakai dan cukup disebutkan di subbab 3.7.4.

In [19]:
VARIAN_ULANG = "large-v3-turbo"      # ganti sesuai varian yang gagal
model_id = dict(VARIAN_DIUJI)[VARIAN_ULANG]

skenario = [
    dict(batch=BATCH_SIZE, grad_ckpt=True,  optim8bit=False),
    dict(batch=2,          grad_ckpt=True,  optim8bit=False),
    dict(batch=1,          grad_ckpt=True,  optim8bit=False),
    dict(batch=1,          grad_ckpt=True,  optim8bit=True),
]
print(f"Menguji ulang {VARIAN_ULANG}\n")
print(f"{'Batch':>6}{'GradCkpt':>10}{'8bit':>7}{'Status':>14}{'Memori':>9}{'dtk/lgk':>9}{'Jam':>8}")
print("-" * 63)
for s in skenario:
    h = uji_lora(VARIAN_ULANG, model_id, **s)
    if h["status"] == "BERHASIL":
        akum = max(1, (BATCH_SIZE * GRAD_ACCUM) // s["batch"])
        e = ekstrapolasi(h["detik_per_langkah"], batch=s["batch"])
        print(f"{s['batch']:>6}{str(s['grad_ckpt']):>10}{str(s['optim8bit']):>7}"
              f"{h['status']:>14}{h['puncak_memori_gb']:>8.1f}G{h['detik_per_langkah']:>9.2f}"
              f"{e['jam_total']:>8.1f}")
        print(f"\nPakai batch {s['batch']} dengan akumulasi gradien {akum}, "
              f"sehingga batch efektif tetap {BATCH_SIZE * GRAD_ACCUM}.")
        break
    print(f"{s['batch']:>6}{str(s['grad_ckpt']):>10}{str(s['optim8bit']):>7}{h['status']:>14}"
          + "".join(f"{'-':>9}" for _ in range(2)) + f"{'-':>8}")

Menguji ulang large-v3-turbo

 Batch  GradCkpt   8bit        Status   Memori  dtk/lgk     Jam
---------------------------------------------------------------
     4      True  False      BERHASIL     9.1G    19.42    65.5

Pakai batch 4 dengan akumulasi gradien 1, sehingga batch efektif tetap 4.


## Bagian 13 — Anggaran waktu keseluruhan

Menjumlahkan seluruh biaya komputasi penelitian dari angka yang sudah kamu ukur, ditambah taksiran kerja manual. Berguna untuk memutuskan apakah cakupanmu masih realistis.

In [20]:
_rtf = globals().get("HASIL_RTF", [])
_lora = globals().get("HASIL_LORA", [])
RTF_TERUKUR = {h["varian"]: h["rtf_rerata"]
               for h in _rtf if h.get("status") == "BERHASIL"}
if not RTF_TERUKUR:
    RTF_TERUKUR = {"tiny": .05, "base": .07, "small": .15,
                   "medium": .35, "large-v3-turbo": .20, "large-v3": .60}
    print("[i] Bagian 3 belum dijalankan. Memakai angka contoh; hasilnya kasar.\n")

detik_uji = JAM_UJI * 3600
VARIAN_TERPILIH = "small"        # ganti setelah Tahap 1 selesai

tahap1 = sum(RTF_TERUKUR.get(n, 0) * detik_uji for n, _ in VARIAN_DIUJI) / 3600
tahap2 = RTF_TERUKUR.get(VARIAN_TERPILIH, 0) * detik_uji / 3600
latih  = ([h["jam_total"] for h in _lora
           if h.get("status") == "BERHASIL" and h["varian"] == VARIAN_TERPILIH] or [0])[0]
tahap3 = latih + tahap2

# Kerja manual: koreksi transkrip berjalan sekitar 5 sampai 10 kali durasi audio.
transkrip_min, transkrip_maks = JAM_TOTAL * 5, JAM_TOTAL * 10

print(f"{'Kegiatan':<44}{'Jam':>10}")
print("-" * 55)
print(f"{'Tahap 1: 6 varian atas data uji':<44}{tahap1:>10.1f}")
print(f"{'Tahap 2: varian terpilih + daftar istilah':<44}{tahap2:>10.1f}")
print(f"{'Tahap 3: pelatihan LoRA':<44}{latih:>10.1f}")
print(f"{'Tahap 3: inferensi model hasil LoRA':<44}{tahap2:>10.1f}")
print(f"{'Pengulangan pengukuran RTF (x' + str(ULANG_RTF) + ')':<44}"
      f"{tahap1*(ULANG_RTF-1):>10.1f}")
print("-" * 55)
komputasi = tahap1 * ULANG_RTF + tahap2 * 2 + latih
print(f"{'SUBTOTAL komputasi':<44}{komputasi:>10.1f}")
print(f"{'Koreksi transkrip acuan 12 jam (manual)':<44}"
      f"{transkrip_min:>7.0f}-{transkrip_maks:.0f}")
print(f"{'Penyusunan glosarium 6 mata kuliah (manual)':<44}{'10-20':>10}")
print("-" * 55)
print(f"{'TOTAL':<44}{komputasi+transkrip_min:>7.0f}-{komputasi+transkrip_maks+20:.0f}")

print("\nPerhatikan komposisinya. Beban terbesar ada pada koreksi transkrip acuan,")
print("bukan pada komputasi. Kalau waktumu sempit, yang dipangkas sebaiknya jumlah")
print("mata kuliah, bukan cakupan rumusan masalah.")
print(f"\nTurun ke 4 mata kuliah (sekitar 8 jam rekaman) menghemat sekitar "
      f"{(JAM_TOTAL-8)*5:.0f} sampai {(JAM_TOTAL-8)*10:.0f} jam kerja manual,")
print("dan tidak menyentuh satu pun rumusan masalah.")

Kegiatan                                           Jam
-------------------------------------------------------
Tahap 1: 6 varian atas data uji                    1.4
Tahap 2: varian terpilih + daftar istilah          0.1
Tahap 3: pelatihan LoRA                            2.8
Tahap 3: inferensi model hasil LoRA                0.1
Pengulangan pengukuran RTF (x3)                    2.7
-------------------------------------------------------
SUBTOTAL komputasi                                 7.1
Koreksi transkrip acuan 12 jam (manual)          60-120
Penyusunan glosarium 6 mata kuliah (manual)      10-20
-------------------------------------------------------
TOTAL                                            67-148

Perhatikan komposisinya. Beban terbesar ada pada koreksi transkrip acuan,
bukan pada komputasi. Kalau waktumu sempit, yang dipangkas sebaiknya jumlah
mata kuliah, bukan cakupan rumusan masalah.

Turun ke 4 mata kuliah (sekitar 8 jam rekaman) menghemat sekitar 20 sampai 41 jam ke

## Bagian 14 — Membawa hasilnya ke Bab 3

### Angka yang langsung masuk naskah

| Dari | Ke subbab | Isi |
|---|---|---|
| Bagian 1 | 3.9.1 | Spesifikasi perangkat, plus bukti terukur sanggup atau tidak |
| Bagian 11 | 3.7.4 | Parameter terlatih absolut dan persentase, batch, gradient checkpointing |
| Bagian 11 | 3.8.4 | Waktu pelatihan dan puncak memori — jawaban paruh sumber daya RM3 |
| Bagian 3 | 3.7.2, 3.7.5 | Varian yang lolos syarat RTF, dasar aturan pemilihan |
| Bagian 4 | 3.7.3 | Mekanisme penyuntikan terpilih beserta bukti ujinya |
| Bagian 4 | 3.8.4 | Selisih RTF akibat daftar istilah, mengisi baris dugaan pada Tabel 2.5 |
| Bagian 6 | 3.6 | Aturan normalisasi final dan contoh sebelum-sesudah untuk Tabel 3.3 |
| Bagian 7 | 3.8.2 | Verifikasi implementasi metrik |
| Bagian 5 | 3.4.4 | Rerata durasi segmen dan jumlah segmen latih |
| Bagian 8 | 3.10.2 | Satuan analisis dan jumlah titik data |

### Keputusan yang baru bisa diambil setelah notebook ini jalan

1. **Varian terpilih untuk Tahap 2 dan 3.** Butuh Bagian 3 dan hasil TTER Tahap 1.
2. **Mekanisme penyuntikan daftar istilah.** Butuh Bagian 4. Kalau A gagal, beralih ke C dan Tahap 1 ikut disesuaikan.
3. **Perangkat pelatihan lokal atau sewaan.** Butuh Bagian 11.
4. **Satuan analisis statistik.** Butuh Bagian 8.
5. **Cakupan data tetap enam mata kuliah atau dipangkas.** Butuh Bagian 13.

### Yang tetap tidak dijawab notebook ini

Notebook ini menguji kelayakan, bukan hasil. Apakah LoRA konvergen, berapa WER dan TTER-nya, apakah daftar istilah benar-benar memperbaiki akurasi — semuanya baru terjawab setelah transkrip acuan dan glosarium sungguhan tersedia. Data tiruan di Bagian 10 dan 5 hanya sah untuk mengukur memori dan waktu, sebab bentuk masukan Whisper tetap terlepas dari isi audionya.